In [1]:
import pandas, numpy, matplotlib, seaborn, pymongo, mysql.connector, faker
print("All libraries imported successfully")


All libraries imported successfully


# Semana 2: Gestión de Datos con Python, SQL y MongoDB

## 1. Generación de Datos con Faker

In [2]:
from faker import Faker
import random
import pandas as pd

fake = Faker('es_MX')

def generar_empleados(n=10):
    datos = []
    departamentos = ['TI', 'RRHH', 'Ventas', 'Finanzas', 'Marketing']
    for _ in range(n):
        registro = {
            "nombre": fake.name(),
            "departamento": random.choice(departamentos),
            "salario": random.randint(30000, 70000),
            "ciudad": fake.city(),
            "email": fake.email()
        }
        datos.append(registro)
    return datos

mis_empleados = generar_empleados(10)
df = pd.DataFrame(mis_empleados)
print("Primeros registros generados:")
print(df.head())

Primeros registros generados:
                    nombre departamento  salario                    ciudad  \
0          Ing. Ilse Ortiz       Ventas    47238             Nueva Comoras   
1          Sra. Tania Leal    Marketing    63861      San Amanda los altos   
2             Jaime Rivero    Marketing    44226      San Amador los bajos   
3  Miguel Eugenio Anguiano    Marketing    41023  San Esther de la Montaña   
4             Evelio Rubio       Ventas    45350              Vieja Mónaco   

                       email  
0       krosales@example.net  
1  salinasofelia@example.org  
2       vpacheco@example.net  
3   diegoarevalo@example.com  
4     bustosjuan@example.net  


## 2. Persistencia en MongoDB (NoSQL)

In [3]:
from pymongo import MongoClient
import json

def conectar_mongodb():
    cliente = MongoClient('mongodb://localhost:27017/')
    db = cliente['empresa_db']
    coleccion = db['empleados']
    return cliente, coleccion

try:
    cliente, coleccion = conectar_mongodb()
    # Limpiar colección e insertar nuevos datos
    coleccion.delete_many({})
    resultado = coleccion.insert_many(mis_empleados)
    print(f"Se insertaron {len(resultado.inserted_ids)} documentos en MongoDB.")
except Exception as e:
    print(f"Error de conexión a MongoDB: {e}")

Se insertaron 10 documentos en MongoDB.


## 3. Operaciones CRUD en MongoDB

In [4]:
# READ: Consultar empleados de un departamento específico (ej. TI)
departamento_interes = 'TI'
query = {'departamento': departamento_interes}
empleados_ti = list(coleccion.find(query))

print(f"Empleados encontrados en departamento {departamento_interes}: {len(empleados_ti)}")
for e in empleados_ti:
    print(f" - {e['nombre']} (${e['salario']})")

Empleados encontrados en departamento TI: 2
 - Julio Gallegos ($30374)
 - Gregorio Hugo de la Torre ($33432)


In [5]:
# UPDATE: Actualizar el salario de un empleado
if empleados_ti:
    nombre_a_actualizar = empleados_ti[0]['nombre']
    nuevo_salario = 75000
    
    coleccion.update_one({'nombre': nombre_a_actualizar}, {'$set': {'salario': nuevo_salario}})
    empleado_act = coleccion.find_one({'nombre': nombre_a_actualizar})
    print(f"Empleado {nombre_a_actualizar} actualizado: ${empleado_act['salario']}")
else:
    print("No hay empleados en TI para actualizar en esta ejecución.")

Empleado Julio Gallegos actualizado: $75000


In [6]:
# DELETE: Eliminar un registro
conteo_antes = coleccion.count_documents({})
coleccion.delete_one({'departamento': 'Ventas'})
conteo_despues = coleccion.count_documents({})

print(f"Control de eliminación: Antes {conteo_antes} | Después {conteo_despues}")

Control de eliminación: Antes 10 | Después 9


## 4. Persistencia en MySQL (Opcional)

In [7]:
import mysql.connector

def insertar_sql(datos):
    try:
        connection = mysql.connector.connect(
            host='localhost',
            database='test_db',
            user='root',
            password=''
        )
        cursor = connection.cursor()
        cursor.execute("CREATE TABLE IF NOT EXISTS empleados_sql (id INT AUTO_INCREMENT PRIMARY KEY, nombre VARCHAR(255), departamento VARCHAR(100), salario INT)")

        query = "INSERT INTO empleados_sql (nombre, departamento, salario) VALUES (%s, %s, %s)"
        valores = [(d["nombre"], d["departamento"], d["salario"]) for d in datos]
        cursor.executemany(query, valores)
        connection.commit()
        print(f"MySQL: Se insertaron {len(datos)} registros en 'empleados_sql'.")
        connection.close()
    except Exception as e:
        print(f"Servicio MySQL no detectado o Error: {e}")

insertar_sql(mis_empleados)

MySQL: Se insertaron 10 registros en 'empleados_sql'.
